# DynaICL Stage 0.6 - Phase D: operational qualification and PILOT (Google Colab, CPU only)

**This notebook can not start the confirmatory run: it contains no such cell.** Seeds used: pilot only (62001, 62101, 62999).
Confirmatory seeds 64101, 64102 and 64999 are never passed to anything here (the harness refuses them).

* Frozen snapshot sha256: `4114fa43cf737c643c341cccd4026d6fcb54b460bfb12ac4a8a2de8352b492e5`
* Frozen manifest sha256: `461dfec8144104f4d35a513dbf1411041494876331f8f969dda4a9cea09be0c0` (compare with the value you stored outside the workspace)
* Pilot bundle (harness; NOT frozen) sha256: `fe122d82f24154e78ffca01f008d2b66c5d3e5ac28e4eba0115ab268e06eaea6`

**Order.** Run the cells top to bottom. Total about 1.5-3 h on 2 vCPU (estimate). The long cells print progress. The only
manual step is the **runtime restart test** (section 9): run part A, then disconnect/delete the runtime when told to, reconnect,
rerun sections 1-3, run part B.

**What to send back:** the final cell writes `phaseD_colab_results.tar.gz` (also on Drive). Everything in it is pilot-labelled.

## 0. Hardware check (a GPU must NOT be attached; Runtime -> Change runtime type -> CPU)

In [ ]:
!nproc
!lscpu | grep -E "Model name|^CPU\(s\)|Thread|Core|MHz|L2|L3"
!free -h
!df -h /content /content/drive 2>/dev/null
!nvidia-smi 2>&1 | head -3
!cat /proc/uptime

## 1. Mount Drive, get the frozen snapshot and the pilot bundle, verify hashes

In [ ]:
import hashlib, os, zipfile, shutil
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/dynaicl_stage06_phaseD", exist_ok=True)
EXPECTED = {"snapshot": "4114fa43cf737c643c341cccd4026d6fcb54b460bfb12ac4a8a2de8352b492e5", "bundle": "fe122d82f24154e78ffca01f008d2b66c5d3e5ac28e4eba0115ab268e06eaea6"}
MANIFEST_SHA = "461dfec8144104f4d35a513dbf1411041494876331f8f969dda4a9cea09be0c0"
os.environ["MANIFEST_SHA"] = MANIFEST_SHA
def sha(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()
for name in ("stage06_frozen_snapshot.zip", "phase_d_pilot_bundle.zip"):
    on_drive = f"/content/drive/MyDrive/dynaicl_stage06_phaseD/" + name
    if os.path.exists(on_drive):
        shutil.copy(on_drive, "/content/" + name)            # a restarted runtime finds them on Drive
    else:
        from google.colab import files
        print("upload", name); files.upload()
        shutil.copy("/content/" + name, on_drive)
assert sha("/content/stage06_frozen_snapshot.zip") == EXPECTED["snapshot"], "snapshot hash mismatch"
assert sha("/content/phase_d_pilot_bundle.zip") == EXPECTED["bundle"], "bundle hash mismatch"
shutil.rmtree("/content/DynaICL", ignore_errors=True); shutil.rmtree("/content/phase_d", ignore_errors=True)
zipfile.ZipFile("/content/stage06_frozen_snapshot.zip").extractall("/content/DynaICL")
zipfile.ZipFile("/content/phase_d_pilot_bundle.zip").extractall("/content/phase_d")
os.chdir("/content/DynaICL")
print("hashes OK; frozen manifest file sha256:", sha("/content/DynaICL/docs/FREEZE_MANIFEST_STAGE06.json"))

## 2. Install the pinned stack (pip warnings about other Colab packages are harmless)

In [ ]:
!pip install -q -r /content/DynaICL/requirements.txt
!python3 -c "import numpy, scipy, yaml; print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| pyyaml', yaml.__version__)"

## 3. Integrity bookend (start), environment report, storage probes

In [ ]:
%cd /content/DynaICL
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py integrity --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --label colab_bookend_start
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py env --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --durable /content/drive/MyDrive/dynaicl_stage06_phaseD --tag colab
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py storage-probe --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --dir /content --tag colab_local_disk
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py storage-probe --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --dir /content/drive/MyDrive/dynaicl_stage06_phaseD --tag colab_drive

## 4. Environment qualification with the FROZEN runner

Colab's Python/packages differ from the frozen environment manifest, so the runner must refuse until a passing qualification
report exists for THIS environment, THIS manifest and THIS config. Reports are written to Drive, outside the frozen set.
`qualify` runs V4 (full Tier-1), V6, V11 and V13 (about 5-10 min each on 2 vCPU).

In [ ]:
%cd /content/DynaICL
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python3 -m dynaicl.experiments.stage06_probe verify --config config/stage06_pilot.yaml --manifest docs/FREEZE_MANIFEST_STAGE06.json --expected-manifest-sha256 $MANIFEST_SHA ; echo "exit code (3 = refused, expected: no qualification yet): $?"
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python3 -m dynaicl.experiments.stage06_probe qualify --config config/stage06_pilot.yaml --manifest docs/FREEZE_MANIFEST_STAGE06.json --expected-manifest-sha256 $MANIFEST_SHA --out /content/drive/MyDrive/dynaicl_stage06_phaseD/qualification_pilot_colab.json
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 python3 -m dynaicl.experiments.stage06_probe qualify --config config/stage06.yaml --manifest docs/FREEZE_MANIFEST_STAGE06.json --expected-manifest-sha256 $MANIFEST_SHA --out /content/drive/MyDrive/dynaicl_stage06_phaseD/qualification_confirmatory_config_colab.json
!python3 -m dynaicl.experiments.stage06_probe verify --config config/stage06_pilot.yaml --manifest docs/FREEZE_MANIFEST_STAGE06.json --expected-manifest-sha256 $MANIFEST_SHA --qualification /content/drive/MyDrive/dynaicl_stage06_phaseD/qualification_pilot_colab.json ; echo "exit code (0 = accepted): $?"
!python3 -m dynaicl.experiments.stage06_probe verify --config config/stage06_pilot.yaml --manifest docs/FREEZE_MANIFEST_STAGE06.json --expected-manifest-sha256 $MANIFEST_SHA --qualification /content/drive/MyDrive/dynaicl_stage06_phaseD/qualification_confirmatory_config_colab.json ; echo "exit code (3 = refused: report is bound to the other config): $?"
!sha256sum /content/drive/MyDrive/dynaicl_stage06_phaseD/qualification_*.json

In [ ]:
%cd /content/DynaICL
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py qualification-gate --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --tag colab

## 5. Reference pilot runs of the frozen runner (frozen pilot config; N = 200; pilot seeds)

`ref_drive` keeps checkpoints on Drive (the durable storage you would use); `ref_local` keeps them on the local disk. Both must be
bitwise identical.

In [ ]:
%cd /content/DynaICL
Q = "/content/drive/MyDrive/dynaicl_stage06_phaseD/qualification_pilot_colab.json"
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py ref-run --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --tag ref_drive --workers $(nproc) --qualification $Q
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py ref-run --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/pilot_work_local --tag ref_local --workers $(nproc) --qualification $Q
!diff -rq /content/drive/MyDrive/dynaicl_stage06_phaseD/work/ref_drive/ckpt /content/pilot_work_local/ref_local/ckpt && echo "ref_drive == ref_local (file-level identical)"

## 6. Interruption and resume (clean interrupt, SIGKILL, self-kill at write stages) on Drive checkpoints

In [ ]:
%cd /content/DynaICL
REF = "/content/drive/MyDrive/dynaicl_stage06_phaseD/work/ref_drive/ckpt"
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py interrupt-tests --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --tag colab --ref-parent $REF --qualification $Q

## 7. Corrupted / truncated checkpoint handling (on copies)

In [ ]:
%cd /content/DynaICL
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py corruption-tests --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --tag colab --ref-parent $REF --qualification $Q

## 8. Determinism across thread counts, profile, scaling and confirmatory-size anchors

Run on the LOCAL disk (`/content`) so timings are not dominated by Drive. Do not run anything else at the same time.
`anchor` runs two real units at N = 4000 on pilot seeds, plus one noise-naive unit at 4000 systems (about 10-25 min total).

In [ ]:
%cd /content/DynaICL
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py determinism --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/pilot_work_local --tag colab --ref-parent $REF
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py profile-units --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/pilot_work_local --tag colab
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py scaling --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/pilot_work_local --tag colab
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py size-sweep --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/pilot_work_local --tag colab --workers $(nproc)
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py anchor --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/pilot_work_local --tag colab --units primary,0,0,2 primary,0,3,0 naive,0,0,2
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py analysis-cost --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/pilot_work_local --tag colab

## 9. Runtime-restart test (durable storage only)

**Part A** starts the frozen pilot run on Drive and waits until 20 units are saved; then it prints READY. At that moment do
`Runtime -> Disconnect and delete runtime`, reconnect (a new VM, possibly a different CPU), rerun sections 1-3 (the zips are on Drive) and
then run **Part B**. The harness logs the CPU model in both phases.

In [ ]:
%cd /content/DynaICL
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py restart-part-a --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --durable /content/drive/MyDrive/dynaicl_stage06_phaseD --workers $(nproc) --qualification $Q

In [ ]:
# PART B: run AFTER the runtime was deleted and sections 1-3 were rerun
%cd /content/DynaICL
Q = "/content/drive/MyDrive/dynaicl_stage06_phaseD/qualification_pilot_colab.json"
REF = "/content/drive/MyDrive/dynaicl_stage06_phaseD/work/ref_drive/ckpt"
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py restart-part-b --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --durable /content/drive/MyDrive/dynaicl_stage06_phaseD --workers $(nproc) --ref-parent $REF --tag colab --qualification $Q

## 10. Launch protection, completeness, structural analysis, seed search, final integrity, archive

In [ ]:
%cd /content/DynaICL
Q = "/content/drive/MyDrive/dynaicl_stage06_phaseD/qualification_pilot_colab.json"
REF = "/content/drive/MyDrive/dynaicl_stage06_phaseD/work/ref_drive/ckpt"
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py launch-protection --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --tag colab --ref-parent $REF
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py completeness --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --tag colab --ref-parent $REF
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py structural-analysis --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --tag colab --ref-parent $REF
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py seed-search --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --tag colab --dirs /content/drive/MyDrive/dynaicl_stage06_phaseD /content/pilot_work_local
!python3 /content/phase_d/phase_d_pilot/pilot_harness.py integrity --frozen-root /content/DynaICL --snapshot /content/stage06_frozen_snapshot.zip --manifest-sha $MANIFEST_SHA --work /content/drive/MyDrive/dynaicl_stage06_phaseD/work --label colab_bookend_end

In [ ]:
import glob, os, tarfile
out = "/content/phaseD_colab_results.tar.gz"
with tarfile.open(out, "w:gz") as tf:
    for d in ("/content/drive/MyDrive/dynaicl_stage06_phaseD/work/reports", "/content/pilot_work_local/reports"):
        if os.path.isdir(d): tf.add(d, arcname=os.path.basename(os.path.dirname(d)) + "_" + os.path.basename(d))
    for f in glob.glob("/content/drive/MyDrive/dynaicl_stage06_phaseD/qualification_*.json") + glob.glob("/content/drive/MyDrive/dynaicl_stage06_phaseD/work/restart_colab/b.log") + glob.glob("/content/drive/MyDrive/dynaicl_stage06_phaseD/work/restart_colab/a.log"):
        tf.add(f, arcname="extra/" + os.path.basename(f))
os.system(f"cp {out} /content/drive/MyDrive/dynaicl_stage06_phaseD/")
from google.colab import files
files.download(out)
print("DO NOT open PILOT_ONLY_do_not_open_effects.json; it is not in the archive.")